# Diagnóstico — performance e semântica HTML dos sites públicos da UFRN (S13)

Plano de análise fixado no **ADR 013** (`docs/decisoes/013-plano-de-analise-estatistica.md`) antes de olhar os resultados.

- **Entrada:** `data/processed/dataset.csv` (tag `v0.2-analise`) e `data/processed/sites/<id>/metrics.json` (só para o top 10 de auditorias). Nada em `data/raw/` é lido.
- **Saídas:** tabelas em `docs/resultados/tabelas/` (CSV) e figuras em `docs/resultados/figuras/` (PNG), com `docs/resultados/tabelas/meta.json` (regra 10).
- **Recortes (ADR 013, 1c):** performance = conjunto principal; semântica = conjunto principal com semântica `ok` e índice não nulo. Cada teste roda **com** e **sem** os sites com `rendered_com_refresh`.
- **Testes:** Spearman com correção de Holm por família; Kruskal-Wallis por tecnologia (grupos com menos de 10 sites unidos em `outro`), ε², Dunn com Holm se significativo. α = 0,05, bicaudal.

Executar da raiz do repo: `python -m jupyter nbconvert --to notebook --execute --inplace analysis/diagnostico.ipynb`.

In [1]:
import hashlib, json, platform, subprocess, sys
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import scikit_posthocs as sp
from scipy import stats

ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
DATASET = ROOT / "data" / "processed" / "dataset.csv"
SITES = ROOT / "data" / "processed" / "sites"
TAB = ROOT / "docs" / "resultados" / "tabelas"
FIG = ROOT / "docs" / "resultados" / "figuras"
TAB.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

ALFA = 0.05
MIN_GRUPO = 10  # ADR 013, 3c

def git(*args):
    try:
        return subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True, check=True).stdout.strip()
    except Exception:
        return None

sha_dataset = hashlib.sha256(DATASET.read_bytes()).hexdigest()
meta = {
    "generatedAt": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "toolVersions": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scipy": scipy.__version__,
        "scikit-posthocs": getattr(sp, "__version__", None),
        "matplotlib": matplotlib.__version__,
    },
    "gitCommit": git("rev-parse", "HEAD"),
    "entrada": {"dataset": "data/processed/dataset.csv", "sha256": sha_dataset},
}
meta

{'generatedAt': '2026-11-12T16:35:53+00:00',
 'toolVersions': {'python': '3.13.5',
  'pandas': '2.3.1',
  'numpy': '2.2.6',
  'scipy': '1.16.3',
  'scikit-posthocs': '0.17.1',
  'matplotlib': '3.10.5'},
 'gitCommit': 'cdc2a68158d4b93d5c7aa901a969c50429635297',
 'entrada': {'dataset': 'data/processed/dataset.csv',
  'sha256': 'f00b6dd951c958c4093998b442d83c1719f6a0ac7ba8c9da1c7a438cbfe48775'}}

In [2]:
df = pd.read_csv(DATASET)
for c in ["conjunto_principal", "meta_refresh", "rendered_com_refresh"]:
    df[c] = df[c].astype(str).str.lower().eq("true")

principal = df[df.conjunto_principal].copy()
semantico = principal[(principal.status_semantica == "ok") & principal.indice_semantica.notna()].copy()

# ADR 013, 3c: grupos de tecnologia com menos de MIN_GRUPO sites viram "outro"
cont_tec = principal.tecnologia.value_counts()
pequenos = cont_tec[cont_tec < MIN_GRUPO].index.tolist()
for d in (principal, semantico):
    d["tec_grupo"] = d.tecnologia.where(~d.tecnologia.isin(pequenos), "outro")

def variantes(d):
    # ADR 013, 1c: com e sem os stubs de meta refresh
    return {"com_refresh": d, "sem_refresh": d[~d.rendered_com_refresh]}

print("dataset:", len(df), "| principal:", len(principal), "| semântico:", len(semantico),
      "| semântico sem refresh:", len(variantes(semantico)["sem_refresh"]))
print("tecnologia (principal):", cont_tec.to_dict(), "| unidos em 'outro':", pequenos)

dataset: 242 | principal: 217 | semântico: 213 | semântico sem refresh: 204
tecnologia (principal): {'estatico': 116, 'wordpress': 45, 'framework-js': 45, 'outro-cms': 9, 'joomla': 2} | unidos em 'outro': ['outro-cms', 'joomla']


## 1. Descritivas

In [3]:
def descritivas(d_perf, d_sem):
    lm = d_sem
    sem_main_nem_nav = int(((lm.lm_main == 0) & (lm.lm_navigation == 0)).sum())
    linhas = [
        ("n_conjunto_principal", len(d_perf)),
        ("n_recorte_semantico", len(d_sem)),
        ("sem_main", int((lm.lm_main == 0).sum())),
        ("sem_nav", int((lm.lm_navigation == 0).sum())),
        ("sem_main_nem_nav", sem_main_nem_nav),
        ("pct_sem_main_nem_nav", 100 * sem_main_nem_nav / len(lm)),
        ("sem_h1", int((lm.titulos_h1 == 0).sum())),
        ("sem_lang", int(lm.lang.isna().sum())),
        ("proporcao_div_span_media", lm.proporcao_div_span.mean()),
        ("proporcao_div_span_mediana", lm.proporcao_div_span.median()),
        ("proporcao_div_span_dp", lm.proporcao_div_span.std()),
        ("indice_mediana", lm.indice_semantica.median()),
        ("indice_q1", lm.indice_semantica.quantile(.25)),
        ("indice_q3", lm.indice_semantica.quantile(.75)),
        ("nota_performance_mediana", d_perf.nota_performance.median()),
        ("nota_performance_q1", d_perf.nota_performance.quantile(.25)),
        ("nota_performance_q3", d_perf.nota_performance.quantile(.75)),
        ("nota_accessibility_mediana", d_perf.nota_accessibility.median()),
        ("lcp_ms_mediana", d_perf.lcp_ms.median()),
        ("peso_total_kib_mediana", d_perf.peso_total_bytes.median() / 1024),
    ]
    return dict(linhas)

vp, vs = variantes(principal), variantes(semantico)
desc = pd.DataFrame({v: descritivas(vp[v], vs[v]) for v in vp})
desc.index.name = "medida"
desc.to_csv(TAB / "descritivas.csv", float_format="%.4f")
desc

,com_refresh,sem_refresh
medida,,
n_conjunto_principal,217.000000,208.000000
n_recorte_semantico,213.000000,204.000000
sem_main,119.000000,117.000000
sem_nav,63.000000,54.000000
sem_main_nem_nav,46.000000,44.000000
pct_sem_main_nem_nav,21.596244,21.568627
sem_h1,76.000000,74.000000
sem_lang,46.000000,44.000000
proporcao_div_span_media,0.403576,0.405158


## 2. Correlações de Spearman (ADR 013, item 2)

Holm aplicado dentro de cada família e de cada variante. Testes com n < 10 ou com uma variável constante não são calculados (ficam registrados com `motivo`) e não entram na correção.

In [4]:
def holm(pvals):
    # Holm (1979): p ajustado = max acumulado de (m - i) * p_(i), limitado a 1
    p = np.asarray(pvals, dtype=float)
    m = len(p)
    ordem = np.argsort(p)
    aj = np.empty(m)
    acc = 0.0
    for i, k in enumerate(ordem):
        acc = max(acc, (m - i) * p[k])
        aj[k] = min(acc, 1.0)
    return aj

# conferência com um caso calculado à mão: p = [0.01, 0.04, 0.03] → [0.03, 0.06, 0.06]
assert np.allclose(holm([0.01, 0.04, 0.03]), [0.03, 0.06, 0.06])

SUBNOTAS = ["sub_landmarks", "sub_titulos", "sub_imagens", "sub_campos", "sub_div_span",
            "sub_metadados", "sub_interativos", "sub_obsoletos"]
FAMILIAS = {
    "principal": [("indice_semantica", "nota_accessibility"), ("indice_semantica", "nota_performance")],
    "secundaria": [(s, "nota_performance") for s in SUBNOTAS]
                  + [("elementos_body", "nota_performance")],
    # proporcao_div_span fica de fora: sub_div_span = 100 * (1 - proporcao_div_span), mesmo teste (ADR 013)
}

linhas = []
for variante, d in variantes(semantico).items():
    for familia, pares in FAMILIAS.items():
        bloco = []
        for x, y in pares:
            par = d[[x, y]].dropna()
            r = {"variante": variante, "familia": familia, "x": x, "y": y, "n": len(par),
                 "rho": np.nan, "p": np.nan, "motivo": ""}
            if len(par) < 10:
                r["motivo"] = "n<10"
            elif par[x].nunique() < 2 or par[y].nunique() < 2:
                r["motivo"] = "constante"
            else:
                rho, p = stats.spearmanr(par[x], par[y])
                r["rho"], r["p"] = rho, p
            bloco.append(r)
        calc = [r for r in bloco if not r["motivo"]]
        for r, pa in zip(calc, holm([r["p"] for r in calc])):
            r["p_holm"] = pa
            r["significativo"] = pa < ALFA
        linhas += bloco

corr = pd.DataFrame(linhas)
corr.to_csv(TAB / "spearman.csv", index=False, float_format="%.6g")
corr

,variante,familia,x,y,n,rho,p,motivo,p_holm,significativo
0,com_refresh,principal,indice_semantica,nota_accessibility,213,0.469487,4.488908e-13,,8.977817e-13,True
1,com_refresh,principal,indice_semantica,nota_performance,213,-0.012253,8.589001e-01,,8.589001e-01,False
2,com_refresh,secundaria,sub_landmarks,nota_performance,213,-0.147560,3.134231e-02,,1.567115e-01,False
3,com_refresh,secundaria,sub_titulos,nota_performance,213,0.086079,2.108564e-01,,8.338273e-01,False
4,com_refresh,secundaria,sub_imagens,nota_performance,199,0.089549,2.084568e-01,,8.338273e-01,False
5,com_refresh,secundaria,sub_campos,nota_performance,108,-0.031848,7.435133e-01,,8.338273e-01,False
6,com_refresh,secundaria,sub_div_span,nota_performance,213,0.218172,1.355030e-03,,9.485212e-03,True
7,com_refresh,secundaria,sub_metadados,nota_performance,213,0.063351,3.575396e-01,,8.338273e-01,False
8,com_refresh,secundaria,sub_interativos,nota_performance,5,NaN,NaN,n<10,NaN,NaN
9,com_refresh,secundaria,sub_obsoletos,nota_performance,213,-0.171190,1.234107e-02,,7.404641e-02,False


## 3. Comparação por tecnologia — Kruskal-Wallis (ADR 013, item 3)

ε² = (H − k + 1) / (n − k) (Tomczak & Tomczak, 2014 — conferir). Dunn com Holm só se p < α.

In [5]:
def kruskal(d, var):
    dd = d[["tec_grupo", var]].dropna()
    grupos = {g: s[var].values for g, s in dd.groupby("tec_grupo")}
    H, p = stats.kruskal(*grupos.values())
    n, k = len(dd), len(grupos)
    eps2 = (H - k + 1) / (n - k)
    med = {f"mediana_{g}": float(np.median(v)) for g, v in grupos.items()}
    ns = {f"n_{g}": len(v) for g, v in grupos.items()}
    return {"n": n, "k": k, "H": H, "p": p, "epsilon2": eps2, **ns, **med}, dd

kw, dunn = [], []
for var, base in [("indice_semantica", semantico), ("nota_performance", principal)]:
    for variante, d in variantes(base).items():
        r, dd = kruskal(d, var)
        kw.append({"variavel": var, "variante": variante, **r})
        if r["p"] < ALFA:
            m = sp.posthoc_dunn(dd, val_col=var, group_col="tec_grupo", p_adjust="holm")
            for i, a in enumerate(m.index):
                for b in m.columns[i + 1:]:
                    dunn.append({"variavel": var, "variante": variante, "grupo_a": a, "grupo_b": b,
                                 "p_holm": m.loc[a, b], "significativo": m.loc[a, b] < ALFA})

kw = pd.DataFrame(kw)
kw.to_csv(TAB / "kruskal_tecnologia.csv", index=False, float_format="%.6g")
dunn = pd.DataFrame(dunn, columns=["variavel", "variante", "grupo_a", "grupo_b", "p_holm", "significativo"])
dunn.to_csv(TAB / "dunn_tecnologia.csv", index=False, float_format="%.6g")
display(kw)
dunn

,variavel,variante,n,k,H,p,epsilon2,n_estatico,n_framework-js,n_outro,n_wordpress,mediana_estatico,mediana_framework-js,mediana_outro,mediana_wordpress
0,indice_semantica,com_refresh,213,4,16.239396,0.001013,0.063346,112,45,11,45,62.08,68.31,80.21,74.83
1,indice_semantica,sem_refresh,204,4,21.724220,0.000074,0.093621,103,45,11,45,60.83,68.31,80.21,74.83
2,nota_performance,com_refresh,217,4,24.872501,0.000016,0.102688,116,45,11,45,62.50,52.00,72.00,62.00
3,nota_performance,sem_refresh,208,4,25.856090,0.000010,0.112040,107,45,11,45,64.00,52.00,72.00,62.00


,variavel,variante,grupo_a,grupo_b,p_holm,significativo
0,indice_semantica,com_refresh,estatico,framework-js,0.771853,False
1,indice_semantica,com_refresh,estatico,outro,0.050112,False
2,indice_semantica,com_refresh,estatico,wordpress,0.003372,True
3,indice_semantica,com_refresh,framework-js,outro,0.148566,False
4,indice_semantica,com_refresh,framework-js,wordpress,0.122514,False
5,indice_semantica,com_refresh,outro,wordpress,0.771853,False
6,indice_semantica,sem_refresh,estatico,framework-js,0.305386,False
7,indice_semantica,sem_refresh,estatico,outro,0.015856,True
8,indice_semantica,sem_refresh,estatico,wordpress,0.000296,True
9,indice_semantica,sem_refresh,framework-js,outro,0.129250,False


## 4. Auditorias do Lighthouse que mais falham (ADR 013, item 4)

Frequência de sites do conjunto principal que falham em cada auditoria (critério do ADR 009), lida do `metrics.json` da execução usada.

In [6]:
cont = {}
ids_principal = set(principal.id)
for id_ in sorted(ids_principal):
    m = json.loads((SITES / id_ / "metrics.json").read_text(encoding="utf-8"))
    for a in (m.get("performance") or {}).get("auditoriasComFalha") or []:
        c = cont.setdefault(a["id"], {"auditoria": a["id"], "titulo": a.get("titulo"),
                                       "categorias": ",".join(a.get("categorias", [])), "sites": 0})
        c["sites"] += 1

aud = pd.DataFrame(cont.values()).sort_values(["sites", "auditoria"], ascending=[False, True]).reset_index(drop=True)
aud["pct"] = 100 * aud.sites / len(principal)
aud.to_csv(TAB / "auditorias_falha.csv", index=False, float_format="%.2f")
aud.head(10).to_csv(TAB / "auditorias_top10.csv", index=False, float_format="%.2f")
print("n =", len(principal))
aud.head(10)

n = 217


,auditoria,titulo,categorias,sites,pct
0,network-dependency-tree-insight,Network dependency tree,performance,186,85.714286
1,render-blocking-insight,Render-blocking requests,performance,181,83.410138
2,cache-insight,Use efficient cache lifetimes,performance,174,80.184332
3,image-delivery-insight,Improve image delivery,performance,174,80.184332
4,unused-javascript,Reduce unused JavaScript,performance,170,78.341014
5,unused-css-rules,Reduce unused CSS,performance,159,73.271889
6,unsized-images,Image elements do not have explicit `width` an...,performance,158,72.811060
7,color-contrast,Background and foreground colors do not have a...,accessibility,154,70.967742
8,font-display-insight,Font display,performance,133,61.290323
9,link-name,Links do not have a discernible name,accessibility,126,58.064516


## 5. Figuras

In [7]:
plt.rcParams.update({"figure.dpi": 150, "savefig.bbox": "tight", "font.size": 9})
SALVAR = {"metadata": {"Software": None}}  # PNG sem versão embutida: mesma figura → mesmos bytes

fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].hist(semantico.indice_semantica, bins=20, range=(0, 100), color="#4c72b0", edgecolor="white")
ax[0].set(xlabel="Índice de semântica (0–100)", ylabel="Sites", title=f"Índice (n = {len(semantico)})")
ax[1].hist(principal.nota_performance, bins=20, range=(0, 100), color="#dd8452", edgecolor="white")
ax[1].set(xlabel="Nota de performance (Lighthouse, mobile)", title=f"Performance (n = {len(principal)})")
fig.savefig(FIG / "distribuicoes.png", **SALVAR)

fig, ax = plt.subplots(1, 2, figsize=(8, 3.4), sharex=True)
for a, y, rotulo in [(ax[0], "nota_accessibility", "Nota de acessibilidade"), (ax[1], "nota_performance", "Nota de performance")]:
    d = semantico[["indice_semantica", y, "rendered_com_refresh"]].dropna()
    a.scatter(d[~d.rendered_com_refresh].indice_semantica, d[~d.rendered_com_refresh][y], s=10, alpha=.6, label="demais")
    a.scatter(d[d.rendered_com_refresh].indice_semantica, d[d.rendered_com_refresh][y], s=18, marker="x", color="red", label="stub de meta refresh")
    r = corr[(corr.variante == "com_refresh") & (corr.x == "indice_semantica") & (corr.y == y)].iloc[0]
    a.set(xlabel="Índice de semântica", ylabel=rotulo, title=f"ρ = {r.rho:.2f}, p(Holm) = {r.p_holm:.3g}")
ax[0].legend(fontsize=7)
fig.savefig(FIG / "dispersao_indice.png", **SALVAR)

ordem = principal.tec_grupo.value_counts().index.tolist()
fig, ax = plt.subplots(1, 2, figsize=(8, 3.4))
for a, d, var, rotulo in [(ax[0], semantico, "indice_semantica", "Índice de semântica"), (ax[1], principal, "nota_performance", "Nota de performance")]:
    dados = [d[d.tec_grupo == g][var].dropna() for g in ordem]
    a.boxplot(dados, tick_labels=[f"{g}\n(n={len(x)})" for g, x in zip(ordem, dados)])
    a.set(ylabel=rotulo)
    a.tick_params(axis="x", labelsize=7)
fig.savefig(FIG / "boxplot_tecnologia.png", **SALVAR)

top = aud.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 3.6))
cores = top.categorias.map(lambda c: "#4c72b0" if "performance" in c else "#55a868" if "accessibility" in c else "#8172b3")
ax.barh(top.auditoria, top.pct, color=cores)
ax.set(xlabel=f"% dos sites do conjunto principal (n = {len(principal)})", xlim=(0, 100))
fig.savefig(FIG / "auditorias_top10.png", **SALVAR)
plt.close("all")
sorted(p.name for p in FIG.glob("*.png"))

['auditorias_top10.png',
 'boxplot_tecnologia.png',
 'dispersao_indice.png',
 'distribuicoes.png']

In [8]:
meta["saidas"] = {
    "tabelas": sorted(p.name for p in TAB.glob("*.csv")),
    "figuras": sorted(p.name for p in FIG.glob("*.png")),
}
meta["recortes"] = {"dataset": len(df), "conjunto_principal": len(principal), "recorte_semantico": len(semantico),
                    "recorte_semantico_sem_refresh": int((~semantico.rendered_com_refresh).sum()),
                    "tecnologia_unidos_em_outro": pequenos}
(TAB / "meta.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
meta

{'generatedAt': '2026-11-12T16:35:53+00:00',
 'toolVersions': {'python': '3.13.5',
  'pandas': '2.3.1',
  'numpy': '2.2.6',
  'scipy': '1.16.3',
  'scikit-posthocs': '0.17.1',
  'matplotlib': '3.10.5'},
 'gitCommit': 'cdc2a68158d4b93d5c7aa901a969c50429635297',
 'entrada': {'dataset': 'data/processed/dataset.csv',
  'sha256': 'f00b6dd951c958c4093998b442d83c1719f6a0ac7ba8c9da1c7a438cbfe48775'},
 'saidas': {'tabelas': ['auditorias_falha.csv',
   'auditorias_top10.csv',
   'descritivas.csv',
   'dunn_tecnologia.csv',
   'kruskal_tecnologia.csv',
   'spearman.csv'],
  'figuras': ['auditorias_top10.png',
   'boxplot_tecnologia.png',
   'dispersao_indice.png',
   'distribuicoes.png']},
 'recortes': {'dataset': 242,
  'conjunto_principal': 217,
  'recorte_semantico': 213,
  'recorte_semantico_sem_refresh': 204,
  'tecnologia_unidos_em_outro': ['outro-cms', 'joomla']}}